
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - AI Search on Databricks
#### Duration: 20 minutes
## Overview
This lecture covers the theory behind AI Search before you build one in the next demo. We'll explore how embeddings and similarity search power retrieval, how AI Search is organized into endpoints and indexes and the search modes it supports, how agents use AI Search under the hood, and when to choose managed RAG vs. building your own. We'll also cover how Unity Catalog governs the entire pipeline.

## Learning Objectives
By the end of this lecture, you will be able to:
1. Explain how embeddings transform text into searchable vectors
2. Describe how cosine similarity drives retrieval ranking
3. Describe the roles of AI Search endpoints and indexes
4. Compare the index types for providing embeddings (managed, self-managed, direct access)
5. Distinguish semantic, keyword, and hybrid search
6. Compare managed RAG (Knowledge Assistants) and custom RAG approaches
7. Explain how Unity Catalog governs AI Search indexes

## A. Embeddings and AI Search Fundamentals

### A1. What Are Embeddings?

An **embedding** is a calculation (represented as a vector) that is based on the context that a word appears in. Texts with similar meanings produce vectors that are close together in mathematical space.

For example, "cozy apartment near the park" and "comfortable flat close to green space" would have similar embeddings, even though they use completely different words.

For example, when creating an AI Search index with `embedding_model_endpoint_name="databricks-gte-large-en"`, Databricks calls this model on every chunk in your table to produce these vectors. Now let's understand how those vectors are used to find relevant results.

**Key considerations when working with embeddings:**
- **Use the same model** for indexing documents and processing queries, as this ensures they exist in the same vector space. With Genie Agents, this is managed for you, but Databricks has fully support for custom workloads.
- **Respect context windows** - embedding models have token limits (e.g., 512 tokens). Text beyond the limit is silently truncated
- **Dimension trade-offs** - higher dimensions capture more nuance but cost more to store and search. The number of dimensions represents the amount of context the model considers when calculating the embeddings. 

<div class="emb" style="width:100%;margin:0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;color:#0b2026;">
  <style>
    .emb *{box-sizing:border-box;}
    .emb .mono{font-family:ui-monospace,Menlo,Consolas,monospace;}
    .emb .row{display:flex;gap:24px;flex-wrap:wrap;align-items:stretch;width:100%;}
    .emb .card{flex:1 1 420px;min-width:0;background:#fff;border:1.5px solid #C6DCEA;border-radius:14px;padding:16px 18px 18px;box-shadow:0 2px 8px rgba(27,49,57,.07);display:flex;flex-direction:column;overflow:hidden;}
    .emb .accent{height:5px;border-radius:5px;margin:-4px 0 12px;background:#FF5F46;}
    .emb .card.b .accent{background:#4299E0;}
    .emb .inner{padding:0 6px 0;display:flex;flex-direction:column;flex:1 1 auto;}
    .emb .eyebrow{font-size:14px;font-weight:800;letter-spacing:.4px;color:#FF5F46;padding-bottom:12px;margin-bottom:18px;border-bottom:1px solid #EEEDE9;}
    .emb .card.b .eyebrow{color:#2272B4;}
    .emb .body{flex:1 1 auto;display:flex;flex-direction:column;justify-content:center;}
    .emb .model{background:#FFF6F4;border:1px solid #FF5F46;border-radius:6px;text-align:center;padding:12px 10px;margin-bottom:16px;}
    .emb .model .t{font-size:16px;font-weight:800;color:#0b2026;}
    .emb .model .s{font-size:14px;color:#5A6F77;margin-top:4px;}
    .emb .mlabel{font-size:14px;font-weight:700;letter-spacing:.3px;color:#618794;margin-bottom:10px;}
    .emb .models{margin-bottom:20px;}
    .emb .mrow{display:flex;align-items:center;gap:10px;margin-bottom:10px;}
    .emb .mrow:last-child{margin-bottom:0;}
    .emb .dot{flex:none;width:7px;height:7px;border-radius:50%;background:#FF5F46;}
    .emb .dot.ext{background:#618794;}
    .emb .mname{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:14px;color:#0b2026;}
    .emb .mext{font-size:14px;font-style:italic;color:#5A6F77;}
    .emb .lane{display:flex;align-items:center;gap:12px;margin-bottom:12px;}
    .emb .lane:last-child{margin-bottom:0;}
    .emb .word{flex:none;width:72px;text-align:center;font-size:14px;font-weight:700;background:transparent;border:1px solid #DCE0E2;border-radius:6px;padding:8px 0;color:#0b2026;}
    .emb .arw{flex:none;color:#618794;font-size:16px;font-weight:800;}
    .emb .vec{flex:1 1 0;min-width:0;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:14px;background:transparent;border:1px solid #DCE0E2;border-radius:6px;padding:9px 12px;color:#0b2026;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;}
    .emb .plot{width:100%;max-width:520px;margin:0 auto;}
    .emb svg{display:block;width:100%;height:auto;}
    .emb .cap{font-size:14px;color:#5A6F77;line-height:1.5;margin:18px 6px 0;padding-top:14px;border-top:1px solid #EEEDE9;}
    .emb .cap b{color:#0b2026;}
  </style>
  <div class="row">
    <div class="card a">
      <div class="accent"></div>
      <div class="inner">
        <div class="eyebrow">SAMPLE EMBEDDINGS</div>
        <div class="body" style="justify-content:flex-start;">
          <div class="model">
            <div class="t">Embedding model</div>
            <div class="s">a neural network trained on large amounts of text</div>
          </div>
          <div class="mlabel">Examples on Databricks</div>
          <div class="models">
            <div class="mrow"><span class="dot"></span><span class="mname">databricks-gte-large-en</span></div>
            <div class="mrow"><span class="dot"></span><span class="mname">databricks-bge-large-en</span></div>
            <div class="mrow"><span class="dot"></span><span class="mname">databricks-qwen3-embedding-0-6b</span></div>
            <div class="mrow"><span class="dot ext"></span><span class="mext">Bring your own external embedding model</span></div>
          </div>
          <div class="lane">
            <div class="word">cat</div><div class="arw">&rarr;</div>
            <div class="vec">[ 0.12, &minus;0.83, 0.05, 0.44, 0.671, &minus;0.29, 0.38, &minus;0.07, 0.52, 0.16, &minus;0.61, 0.24, 0.09, &minus;0.45, 0.73, 0.31, &hellip; ]</div>
          </div>
          <div class="lane">
            <div class="word">dog</div><div class="arw">&rarr;</div>
            <div class="vec">[ 0.15, &minus;0.79, 0.11, 0.40, 0.63, &minus;0.33, 0.35, &minus;0.04, 0.49, 0.21, &minus;0.58, 0.27, 0.13, &minus;0.41, 0.70, 0.28, &hellip; ]</div>
          </div>
          <div class="lane">
            <div class="word">car</div><div class="arw">&rarr;</div>
            <div class="vec">[ &minus;0.66, 0.28, 0.91, &minus;0.10, &minus;0.47, 0.82, &minus;0.19, 0.55, &minus;0.72, 0.34, 0.08, &minus;0.63, 0.41, 0.77, &minus;0.22, 0.60, &hellip; ]</div>
          </div>
        </div>
      </div>
      <div class="cap">Every item maps to a <b>fixed-length</b> list of numbers (often 1024), regardless of input size. Related inputs receive similar numbers: <span class="mono">cat</span> and <span class="mono">dog</span> are close; <span class="mono">car</span> is not.</div>
    </div>
    <div class="card b">
      <div class="accent"></div>
      <div class="inner">
        <div class="eyebrow">SIMILAR MEANING LANDS NEARBY</div>
        <div class="body">
          <div class="plot">
            <svg viewBox="0 0 400 290" role="img" aria-label="Scatter plot of word embeddings grouped into three clusters: animals, royalty, and vehicles.">
              <defs>
                <marker id="emb-ax" viewBox="0 0 10 10" refX="7" refY="5" markerWidth="6" markerHeight="6" orient="auto"><path d="M0 0 L10 5 L0 10 z" fill="#618794"/></marker>
              </defs>
              <line x1="16" y1="275" x2="392" y2="275" stroke="#618794" stroke-width="1.5" marker-end="url(#emb-ax)"/>
              <line x1="16" y1="275" x2="16" y2="14" stroke="#618794" stroke-width="1.5" marker-end="url(#emb-ax)"/>
              <ellipse cx="118" cy="116" rx="62" ry="52" fill="none" stroke="#FF9E94" stroke-width="1.3" stroke-dasharray="4 4"/>
              <ellipse cx="306" cy="100" rx="56" ry="42" fill="none" stroke="#C4CCD6" stroke-width="1.3" stroke-dasharray="4 4"/>
              <ellipse cx="306" cy="212" rx="58" ry="42" fill="none" stroke="#C4CCD6" stroke-width="1.3" stroke-dasharray="4 4"/>
              <text x="118" y="50" text-anchor="middle" font-size="14" font-weight="800" letter-spacing="0.6" fill="#FF5F46">ANIMALS</text>
              <text x="306" y="44" text-anchor="middle" font-size="14" font-weight="800" letter-spacing="0.6" fill="#1B5162">ROYALTY</text>
              <text x="306" y="160" text-anchor="middle" font-size="14" font-weight="800" letter-spacing="0.6" fill="#2272B4">VEHICLES</text>
              <circle cx="102" cy="118" r="5" fill="#FF5F46"/><text x="92" y="123" text-anchor="end" font-size="14" font-weight="600" fill="#0b2026">cat</text>
              <circle cx="142" cy="92" r="5" fill="#FF5F46"/><text x="152" y="97" font-size="14" font-weight="600" fill="#0b2026">dog</text>
              <circle cx="108" cy="146" r="5" fill="#FF5F46"/><text x="118" y="151" font-size="14" font-weight="600" fill="#0b2026">puppy</text>
              <circle cx="292" cy="94" r="5" fill="#1B5162"/><text x="282" y="99" text-anchor="end" font-size="14" font-weight="600" fill="#0b2026">king</text>
              <circle cx="318" cy="116" r="5" fill="#1B5162"/><text x="328" y="121" font-size="14" font-weight="600" fill="#0b2026">queen</text>
              <circle cx="292" cy="206" r="5" fill="#2272B4"/><text x="282" y="211" text-anchor="end" font-size="14" font-weight="600" fill="#0b2026">car</text>
              <circle cx="322" cy="226" r="5" fill="#2272B4"/><text x="332" y="231" font-size="14" font-weight="600" fill="#0b2026">truck</text>
            </svg>
          </div>
        </div>
      </div>
      <div class="cap">Because meaning is stored as <b>position</b>, related words fall into the same region. Real embeddings span <b>hundreds or thousands of dimensions</b> &mdash; shown here in 2D for intuition.</div>
    </div>
  </div>
</div>

### A2. The Similarity Search Pipeline

When you search a vector index, the system:
1. Converts your query text into an embedding vector
2. Finds the closest vectors in the index (using cosine similarity or other distance metrics)
3. Returns the corresponding text chunks, ranked by relevance

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 200" role="img" style="font-family: sans-serif;">
  <title>Similarity Search Flow</title>
  <defs>
    <marker id="ss-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Query -->
  <rect x="10" y="60" width="130" height="60" rx="8" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="75" y="84" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Query Text</text>
  <text x="75" y="102" text-anchor="middle" font-size="15" fill="#618794">"pet-friendly?"</text>

  <path d="M140 90 L180 90" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ss-arrow)"/>

  <!-- Embed -->
  <rect x="185" y="60" width="130" height="60" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="250" y="84" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Embed</text>
  <text x="250" y="102" text-anchor="middle" font-size="15" fill="#618794">[0.23, -0.41, ...]</text>

  <path d="M315 90 L355 90" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ss-arrow)"/>

  <!-- Search -->
  <rect x="360" y="60" width="130" height="60" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="425" y="84" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Find Nearest</text>
  <text x="425" y="102" text-anchor="middle" font-size="15" fill="#618794">Cosine similarity</text>

  <path d="M490 90 L530 90" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ss-arrow)"/>

  <!-- Results -->
  <rect x="535" y="40" width="130" height="100" rx="8" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5"/>
  <text x="600" y="64" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Top Results</text>
  <rect x="547" y="76" width="106" height="20" rx="4" fill="#2272B4" fill-opacity="0.30"/>
  <text x="600" y="90" text-anchor="middle" font-size="14" fill="#0b2026">Chunk 1 (0.94)</text>
  <rect x="547" y="100" width="106" height="20" rx="4" fill="#2272B4" fill-opacity="0.18"/>
  <text x="600" y="114" text-anchor="middle" font-size="14" fill="#0b2026">Chunk 2 (0.87)</text>
  <rect x="547" y="124" width="106" height="10" rx="4" fill="#2272B4" fill-opacity="0.10"/>
</svg>
</div>

For fast retrieval at scale, Databricks uses **Approximate Nearest Neighbors (ANN)** algorithms, trading a small amount of precision for dramatic speed gains.

When you use `similarity_search` on an AI Search index, your query text is embedded, compared against all vectors in the index, and the top matches are returned with their similarity scores. You'll see this in action in the next demo.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Learn More</strong>
<p style="margin: 8px 0 0 0; color: #333;">
See the full AI Search documentation (<a href="https://docs.databricks.com/aws/en/ai-search/ai-search" style="color: #1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-search/ai-search" style="color: #1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-search/ai-search" style="color: #1976d2;">GCP</a>) for details on endpoints, indexes, and APIs.
</p>
</div>

### A3. What Is AI Search?

**AI Search** is a retrieval solution built into the Databricks Data Intelligence Platform and integrated with Unity Catalog governance. Given a query, its core job is to find the stored embeddings that are **most semantically similar** (using vector similarity). It is the retrieval engine behind RAG systems, recommender systems, and image or video search.

To stay fast at scale, AI Search uses the **Hierarchical Navigable Small World (HNSW)** algorithm for approximate nearest neighbor (ANN) search. This means lookups remain quick even across hundreds of millions of vectors.

Every deployment is built from two components: **endpoints** and **indexes**. We look at those next, then at how embeddings get into an index and the ways you can search it.

### A4. Providing Embeddings: Index Types

How do vectors get into an index? There are three approaches. The first two keep the index automatically in sync with a Delta source; the third gives you manual control. In each diagram, the left side shows how the index is **created** and the right side shows the **query endpoint** at serving time — the numbered steps trace a single query.

**Databricks-managed embeddings (Delta Sync)** — you point at a text column and choose an embedding model; Databricks computes the embeddings, optionally saves them to a table, and keeps the index synced as the source changes. The simplest path, and the one used in this course. At query time, (1) the query is embedded with the same model, (2) compared against the vector index, and (3) the nearest results are returned to the app.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 300" role="img" style="font-family: sans-serif;">
  <title>Databricks-managed embeddings: source Delta table to embedding model to vector index, queried at serving time</title>
  <defs>
    <marker id="a5m-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M1 1L9 5L1 9Z" fill="#1B3139"/>
    </marker>
  </defs>
  <text x="20" y="26" font-size="14" font-weight="700" fill="#618794">Create vector index</text>
  <text x="20" y="44" font-size="14" font-weight="700" fill="#618794">(Databricks calculates and</text>
  <text x="20" y="62" font-size="14" font-weight="700" fill="#618794">optionally saves embeddings)</text>
  <text x="560" y="26" font-size="14" font-weight="700" fill="#618794">Query endpoint</text>
  <line x1="490" y1="18" x2="490" y2="292" stroke="#DCE0E2" stroke-width="1.5"/>
  <!-- Source Delta table -->
  <rect x="15" y="125" width="115" height="56" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.2"/>
  <text x="72" y="149" text-anchor="middle" font-size="13" fill="#0b2026">Source Delta</text>
  <text x="72" y="166" text-anchor="middle" font-size="13" fill="#0b2026">table</text>
  <path d="M130 153 L210 153" fill="none" stroke="#1B3139" stroke-width="1.6" marker-end="url(#a5m-arrow)"/>
  <!-- Embedding model -->
  <rect x="215" y="125" width="140" height="56" rx="2" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="285" y="158" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">Embedding model</text>
  <!-- Embeddings table (optional, dashed) -->
  <path d="M258 181 L122 246" fill="none" stroke="#1B3139" stroke-width="1.6" marker-end="url(#a5m-arrow)"/>
  <rect x="35" y="250" width="130" height="46" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.2" stroke-dasharray="5 4"/>
  <text x="100" y="272" text-anchor="middle" font-size="13" fill="#0b2026">Embeddings</text>
  <text x="100" y="289" text-anchor="middle" font-size="13" fill="#0b2026">table</text>
  <!-- Vector index cylinder (drawn before the incoming arrows so the arrowheads sit on top) -->
  <path d="M236 234 L236 286 A64 10 0 0 1 364 286 L364 234 Z" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <ellipse cx="300" cy="234" rx="64" ry="10" fill="#2272B4" stroke="#2272B4" stroke-width="1.5"/>
  <text x="300" y="268" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">Vector index</text>
  <!-- Embedding model to vector index: solid + dashed(step 2) -->
  <path d="M278 181 L278 230" fill="none" stroke="#1B3139" stroke-width="1.6" marker-end="url(#a5m-arrow)"/>
  <path d="M322 181 L322 230" fill="none" stroke="#1B3139" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#a5m-arrow)"/>
  <!-- Query -->
  <ellipse cx="600" cy="153" rx="72" ry="35" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.8"/>
  <text x="600" y="158" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Query</text>
  <path d="M528 153 L360 153" fill="none" stroke="#1B3139" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#a5m-arrow)"/>
  <!-- LLM application -->
  <rect x="730" y="125" width="110" height="56" rx="2" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="785" y="149" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">LLM</text>
  <text x="785" y="166" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">application</text>
  <path d="M672 153 L726 153" fill="none" stroke="#1B3139" stroke-width="1.6" marker-end="url(#a5m-arrow)"/>
  <!-- step 3: vector index back to query -->
  <path d="M366 266 C 480 300, 556 246, 598 190" fill="none" stroke="#1B3139" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#a5m-arrow)"/>
  <!-- badges -->
  <circle cx="505" cy="139" r="12" fill="#1B3139" stroke="#fff" stroke-width="2"/>
  <text x="505" y="144" text-anchor="middle" font-size="13" font-weight="700" fill="#fff">1</text>
  <circle cx="345" cy="205" r="12" fill="#1B3139" stroke="#fff" stroke-width="2"/>
  <text x="345" y="210" text-anchor="middle" font-size="13" font-weight="700" fill="#fff">2</text>
  <circle cx="522" cy="255" r="12" fill="#1B3139" stroke="#fff" stroke-width="2"/>
  <text x="522" y="260" text-anchor="middle" font-size="13" font-weight="700" fill="#fff">3</text>
</svg>
</div>

**Self-managed embeddings (Delta Sync)** — you precompute embeddings and store them in the source table; the index still auto-syncs on changes, but there is no embedding model on the create side. You **cannot later convert** a self-managed index to managed — you would recreate it.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 240" role="img" style="font-family: sans-serif;">
  <title>Self-managed precalculated embeddings: source Delta table syncs to a vector index that is queried at serving time</title>
  <defs>
    <marker id="a5s-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M1 1L9 5L1 9Z" fill="#1B3139"/>
    </marker>
  </defs>
  <text x="20" y="26" font-size="14" font-weight="700" fill="#618794">Create vector index</text>
  <text x="20" y="44" font-size="14" font-weight="700" fill="#618794">(precalculated embeddings)</text>
  <text x="560" y="26" font-size="14" font-weight="700" fill="#618794">Query endpoint</text>
  <line x1="490" y1="18" x2="490" y2="232" stroke="#DCE0E2" stroke-width="1.5"/>
  <!-- Source Delta table -->
  <rect x="15" y="100" width="115" height="56" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.2"/>
  <text x="72" y="124" text-anchor="middle" font-size="13" fill="#0b2026">Source Delta</text>
  <text x="72" y="141" text-anchor="middle" font-size="13" fill="#0b2026">table</text>
  <path d="M130 128 L212 128" fill="none" stroke="#1B3139" stroke-width="1.6" marker-end="url(#a5s-arrow)"/>
  <!-- Vector index cylinder cx=290 -->
  <path d="M226 108 L226 158 A64 10 0 0 1 354 158 L354 108 Z" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <ellipse cx="290" cy="108" rx="64" ry="10" fill="#2272B4" stroke="#2272B4" stroke-width="1.5"/>
  <text x="290" y="140" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">Vector index</text>
  <!-- Query -->
  <ellipse cx="585" cy="128" rx="72" ry="35" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.8"/>
  <text x="585" y="133" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Query</text>
  <path d="M513 128 L358 128" fill="none" stroke="#1B3139" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#a5s-arrow)"/>
  <!-- LLM application -->
  <rect x="715" y="100" width="110" height="56" rx="2" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="770" y="124" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">LLM</text>
  <text x="770" y="141" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">application</text>
  <path d="M657 128 L711 128" fill="none" stroke="#1B3139" stroke-width="1.6" marker-end="url(#a5s-arrow)"/>
  <!-- step 2: vector index back to query -->
  <path d="M300 168 C 420 224, 520 206, 583 165" fill="none" stroke="#1B3139" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#a5s-arrow)"/>
  <!-- badges -->
  <circle cx="492" cy="114" r="12" fill="#1B3139" stroke="#fff" stroke-width="2"/>
  <text x="492" y="119" text-anchor="middle" font-size="13" font-weight="700" fill="#fff">1</text>
  <circle cx="470" cy="205" r="12" fill="#1B3139" stroke="#fff" stroke-width="2"/>
  <text x="470" y="210" text-anchor="middle" font-size="13" font-weight="700" fill="#fff">2</text>
</svg>
</div>

**Direct Access** — you write vectors directly through the REST API or SDK, with no source table and no automatic sync. Best when an external pipeline computes embeddings or you need real-time upserts.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 760 230" role="img" style="font-family: sans-serif;">
  <title>Direct Access: a vector index with no source table and no automatic sync, queried at serving time</title>
  <defs>
    <marker id="a5d-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M1 1L9 5L1 9Z" fill="#1B3139"/>
    </marker>
  </defs>
  <text x="20" y="26" font-size="14" font-weight="700" fill="#618794">Create vector index</text>
  <text x="20" y="44" font-size="14" font-weight="700" fill="#618794">(precalculated embeddings,</text>
  <text x="20" y="62" font-size="14" font-weight="700" fill="#618794">no automatic sync)</text>
  <text x="470" y="26" font-size="14" font-weight="700" fill="#618794">Query endpoint</text>
  <line x1="380" y1="18" x2="380" y2="222" stroke="#DCE0E2" stroke-width="1.5"/>
  <!-- step 2: vector index back to query (arcs over the top of the cylinder) -->
  <path d="M170 94 C 250 50, 390 50, 440 106" fill="none" stroke="#1B3139" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#a5d-arrow)"/>
  <!-- Vector index cylinder cx=170 -->
  <path d="M106 104 L106 154 A64 10 0 0 1 234 154 L234 104 Z" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <ellipse cx="170" cy="104" rx="64" ry="10" fill="#2272B4" stroke="#2272B4" stroke-width="1.5"/>
  <text x="170" y="136" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">Vector index</text>
  <!-- Query -->
  <ellipse cx="500" cy="126" rx="72" ry="35" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.8"/>
  <text x="500" y="131" text-anchor="middle" font-size="15" font-weight="700" fill="#0b2026">Query</text>
  <path d="M428 126 L238 126" fill="none" stroke="#1B3139" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#a5d-arrow)"/>
  <!-- LLM application -->
  <rect x="630" y="98" width="110" height="56" rx="2" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="685" y="122" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">LLM</text>
  <text x="685" y="139" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">application</text>
  <path d="M572 126 L626 126" fill="none" stroke="#1B3139" stroke-width="1.6" marker-end="url(#a5d-arrow)"/>
  <!-- badges -->
  <circle cx="408" cy="112" r="12" fill="#1B3139" stroke="#fff" stroke-width="2"/>
  <text x="408" y="117" text-anchor="middle" font-size="13" font-weight="700" fill="#fff">1</text>
  <circle cx="319" cy="58" r="12" fill="#1B3139" stroke="#fff" stroke-width="2"/>
  <text x="319" y="63" text-anchor="middle" font-size="13" font-weight="700" fill="#fff">2</text>
</svg>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Full-text search index (Beta)</strong>
<p style="margin: 8px 0 0 0; color: #333;">
Available on <strong>storage-optimized</strong> endpoints only, a full-text search index needs <strong>no embedding column</strong>: it searches text directly with BM25 keyword matching and uses triggered sync. Useful for pure keyword retrieval when vectors are not needed.
</p>
</div>

### A5. CDF for RAG Workloads
A RAG agent is only as good as the index behind it. Source documents changes constantly. For example, policies are revised, tickets arrive, specs evolved, etc. When it comes to RAG workloads, there is always one crucial question that needs answering from an engineering perspective: **which chunks actually changed since the last pipeline run?** Databricks' Change Data Feed (CDF) answers this question by addressing cost and sync modes. 
- **Embeddings can be expensive**: Embeddings require compute or API calls. Without change tracking, you either re-embed the whole corpus or accept a stale nightly batch. CDF turns the work into a function of change volume, not corpus size. 
- **Two sync modes**: A Delta Sync index reads CDF either on demand or as a continuous stream. 

<div class="cdfai">
  <style>
    .cdfai { max-width: 100%; margin: 0 auto; font-family: -apple-system, "Segoe UI", Roboto, sans-serif; color: #0b2026; }
    .cdfai h3 { font-size: 21px; margin: 0 0 10px 0; color: #0b2026; }
    .cdfai h4 { font-size: 17px; margin: 0 0 8px 0; color: #0b2026; }
    .cdfai p { font-size: 15px; line-height: 1.55; margin: 0 0 14px 0; }
    .cdfai .card { background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); margin-bottom: 22px; }
    .cdfai .bar { height: 5px; border-radius: 5px; margin: -4px 0 14px; }
    .cdfai .bar-blue { background: #4299E0; }
    .cdfai .bar-red { background: #FF3621; }
    .cdfai .bar-coral { background: #FF5F46; }
    .cdfai .cap { font-size: 14px; color: #618794; margin: 8px 0 0 0; line-height: 1.5; }
    .cdfai ul { margin: 6px 0 22px 0; padding-left: 22px; }
    .cdfai li { font-size: 15px; line-height: 1.55; margin-bottom: 12px; }
    .cdfai code { font-family: ui-monospace, "Cascadia Code", Consolas, monospace; font-size: 14px; background: rgba(66,153,224,0.10); padding: 1px 5px; border-radius: 4px; }
  </style>
  <div class="card">
    <div class="bar bar-blue"></div>
    <h4>The incremental RAG pipeline</h4>
    <p>CDF emits a row-level change stream (<code>insert</code>, <code>update</code>, <code>delete</code>) from the source Delta table. Every downstream stage &mdash; chunking, embedding, index sync &mdash; consumes only those rows. The numbered steps trace one changed document all the way to the agent.</p>
    <svg width="100%" viewBox="0 0 1060 320" role="img" style="font-family: sans-serif;">
      <title>Change data feed flows from source Delta table through chunking and embedding into a Delta Sync vector index serving a RAG agent</title>
      <defs>
        <marker id="cdfai-arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
          <path d="M1 1L9 5L1 9Z" fill="#0b2026"/>
        </marker>
        <marker id="cdfai-arrc" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
          <path d="M1 1L9 5L1 9Z" fill="#FF5F46"/>
        </marker>
      </defs>
      <text x="148" y="24" text-anchor="middle" font-size="14" font-weight="700" fill="#618794">Source of truth</text>
      <text x="578" y="24" text-anchor="middle" font-size="14" font-weight="700" fill="#618794">Incremental processing (only changed rows)</text>
      <text x="960" y="24" text-anchor="middle" font-size="14" font-weight="700" fill="#618794">Serving</text>
      <line x1="296" y1="34" x2="296" y2="300" stroke="#C6DCEA" stroke-width="2"/>
      <line x1="860" y1="34" x2="860" y2="300" stroke="#C6DCEA" stroke-width="2"/>
      <!-- source documents -->
      <rect x="18" y="60" width="150" height="54" rx="6" fill="#fff" stroke="#0b2026" stroke-width="1.3"/>
      <text x="93" y="83" text-anchor="middle" font-size="14" fill="#0b2026">Documents, tickets,</text>
      <text x="93" y="101" text-anchor="middle" font-size="14" fill="#0b2026">policies, specs</text>
      <path d="M93 116 L93 150" fill="none" stroke="#0b2026" stroke-width="1.6" marker-end="url(#cdfai-arr)"/>
      <!-- delta table -->
      <path d="M22 166 L22 232 A71 11 0 0 0 164 232 L164 166 Z" fill="rgba(66,153,224,0.10)" stroke="#4299E0" stroke-width="1.6"/>
      <ellipse cx="93" cy="166" rx="71" ry="11" fill="#4299E0" stroke="#4299E0" stroke-width="1.6"/>
      <text x="93" y="198" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Source Delta</text>
      <text x="93" y="216" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">table</text>
      <text x="93" y="262" text-anchor="middle" font-size="14" fill="#618794">delta.enableChangeData</text>
      <text x="93" y="280" text-anchor="middle" font-size="14" fill="#618794">Feed = true</text>
      <!-- CDF stream band -->
      <rect x="186" y="150" width="86" height="98" rx="6" fill="rgba(255,95,70,0.12)" stroke="#FF5F46" stroke-width="1.8"/>
      <text x="229" y="176" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">CDF</text>
      <text x="229" y="196" text-anchor="middle" font-size="14" fill="#0b2026">insert</text>
      <text x="229" y="214" text-anchor="middle" font-size="14" fill="#0b2026">update</text>
      <text x="229" y="232" text-anchor="middle" font-size="14" fill="#0b2026">delete</text>
      <path d="M164 199 L182 199" fill="none" stroke="#FF5F46" stroke-width="2" marker-end="url(#cdfai-arrc)"/>
      <path d="M272 199 L316 199" fill="none" stroke="#FF5F46" stroke-width="2" marker-end="url(#cdfai-arrc)"/>
      <!-- chunk -->
      <rect x="320" y="170" width="140" height="58" rx="6" fill="#fff" stroke="#0b2026" stroke-width="1.3"/>
      <text x="390" y="195" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Parse + chunk</text>
      <text x="390" y="215" text-anchor="middle" font-size="14" fill="#618794">changed docs only</text>
      <path d="M460 199 L502 199" fill="none" stroke="#0b2026" stroke-width="1.8" marker-end="url(#cdfai-arr)"/>
      <!-- embed -->
      <rect x="506" y="170" width="150" height="58" rx="6" fill="rgba(66,153,224,0.10)" stroke="#4299E0" stroke-width="1.8"/>
      <text x="581" y="195" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Embedding model</text>
      <text x="581" y="215" text-anchor="middle" font-size="14" fill="#618794">GPU / API $$$</text>
      <path d="M656 199 L700 199" fill="none" stroke="#0b2026" stroke-width="1.8" marker-end="url(#cdfai-arr)"/>
      <!-- vector index -->
      <path d="M704 172 L704 232 A62 10 0 0 0 828 232 L828 172 Z" fill="rgba(66,153,224,0.10)" stroke="#4299E0" stroke-width="1.6"/>
      <ellipse cx="766" cy="172" rx="62" ry="10" fill="#4299E0" stroke="#4299E0" stroke-width="1.6"/>
      <text x="766" y="202" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Delta Sync</text>
      <text x="766" y="220" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">vector index</text>
      <text x="766" y="266" text-anchor="middle" font-size="14" fill="#618794">checkpointed &mdash; no</text>
      <text x="766" y="284" text-anchor="middle" font-size="14" fill="#618794">reload of existing rows</text>
      <!-- agent -->
      <rect x="892" y="100" width="146" height="58" rx="6" fill="rgba(255,95,70,0.12)" stroke="#FF5F46" stroke-width="1.8"/>
      <text x="965" y="125" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Retrieval query</text>
      <text x="965" y="145" text-anchor="middle" font-size="14" fill="#618794">from the agent</text>
      <rect x="892" y="212" width="146" height="58" rx="6" fill="#fff" stroke="#0b2026" stroke-width="1.3"/>
      <text x="965" y="237" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">LLM / agent</text>
      <text x="965" y="257" text-anchor="middle" font-size="14" fill="#618794">grounded answer</text>
      <path d="M888 129 L832 170" fill="none" stroke="#0b2026" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#cdfai-arr)"/>
      <path d="M830 236 L888 241" fill="none" stroke="#0b2026" stroke-width="1.6" stroke-dasharray="5 4" marker-end="url(#cdfai-arr)"/>
      <!-- badges -->
      <circle cx="229" cy="132" r="13" fill="#FF3621"/>
      <text x="229" y="137" text-anchor="middle" font-size="14" font-weight="700" fill="#fff">1</text>
      <circle cx="390" cy="152" r="13" fill="#FF3621"/>
      <text x="390" y="157" text-anchor="middle" font-size="14" font-weight="700" fill="#fff">2</text>
      <circle cx="581" cy="152" r="13" fill="#FF3621"/>
      <text x="581" y="157" text-anchor="middle" font-size="14" font-weight="700" fill="#fff">3</text>
      <circle cx="766" cy="140" r="13" fill="#FF3621"/>
      <text x="766" y="145" text-anchor="middle" font-size="14" font-weight="700" fill="#fff">4</text>
      <circle cx="965" cy="186" r="13" fill="#FF3621"/>
      <text x="965" y="191" text-anchor="middle" font-size="14" font-weight="700" fill="#fff">5</text>
    </svg>

  </div>
  <div class="card">
    <div class="bar bar-red"></div>
    <h4>Deletes and updates prevent hallucination</h4>
    <p>Insert-only pipelines are the hidden failure mode. A retracted policy that lingers in the index will be retrieved and cited as fact.</p>
    <svg width="100%" viewBox="0 0 1060 210" role="img" style="font-family: sans-serif;">
      <title>Without CDF a deleted document remains in the index and is cited; with CDF the delete event removes the chunk</title>
      <defs>
        <marker id="cdfai-arr2" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
          <path d="M1 1L9 5L1 9Z" fill="#0b2026"/>
        </marker>
      </defs>
      <text x="18" y="22" font-size="14" font-weight="700" fill="#FF3621">Insert-only sync</text>
      <rect x="18" y="34" width="250" height="50" rx="6" fill="#fff" stroke="#0b2026" stroke-width="1.3"/>
      <text x="143" y="56" text-anchor="middle" font-size="14" fill="#0b2026">Doc retracted</text>
      <text x="143" y="75" text-anchor="middle" font-size="14" fill="#618794">in source</text>
      <path d="M268 59 L322 59" fill="none" stroke="#0b2026" stroke-width="1.6" marker-end="url(#cdfai-arr2)"/>
      <rect x="326" y="34" width="280" height="50" rx="6" fill="rgba(255,54,33,0.14)" stroke="#FF3621" stroke-width="1.8"/>
      <text x="466" y="56" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Stale chunk stays</text>
      <text x="466" y="75" text-anchor="middle" font-size="14" fill="#0b2026">in the index</text>
      <path d="M606 59 L660 59" fill="none" stroke="#0b2026" stroke-width="1.6" marker-end="url(#cdfai-arr2)"/>
      <rect x="664" y="34" width="250" height="50" rx="6" fill="#fff" stroke="#FF3621" stroke-width="1.6"/>
      <text x="789" y="56" text-anchor="middle" font-size="14" font-weight="700" fill="#FF3621">Agent cites</text>
      <text x="789" y="75" text-anchor="middle" font-size="14" font-weight="700" fill="#FF3621">dead content</text>
      <line x1="18" y1="106" x2="1042" y2="106" stroke="#C6DCEA" stroke-width="2"/>
      <text x="18" y="134" font-size="14" font-weight="700" fill="#4299E0">CDF-driven sync</text>
      <rect x="18" y="146" width="250" height="50" rx="6" fill="#fff" stroke="#0b2026" stroke-width="1.3"/>
      <text x="143" y="168" text-anchor="middle" font-size="14" fill="#0b2026">Doc retracted</text>
      <text x="143" y="187" text-anchor="middle" font-size="14" fill="#618794">in source</text>
      <path d="M268 171 L322 171" fill="none" stroke="#0b2026" stroke-width="1.6" marker-end="url(#cdfai-arr2)"/>
      <rect x="326" y="146" width="280" height="50" rx="6" fill="rgba(255,95,70,0.12)" stroke="#FF5F46" stroke-width="1.8"/>
      <text x="466" y="168" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">delete /</text>
      <text x="466" y="187" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">update_postimage</text>
      <path d="M606 171 L660 171" fill="none" stroke="#0b2026" stroke-width="1.6" marker-end="url(#cdfai-arr2)"/>
      <rect x="664" y="146" width="250" height="50" rx="6" fill="rgba(66,153,224,0.10)" stroke="#4299E0" stroke-width="1.8"/>
      <text x="789" y="168" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Chunk removed</text>
      <text x="789" y="187" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">or replaced</text>
    </svg>

  </div>
  <div class="card">
    <div class="bar bar-coral"></div>
    <h4>Lakebase CDF: operational data straight into the agent</h4>
    <p>Operational Postgres state &mdash; customer records, order history, live catalogs &mdash; streams into Delta with roughly 15-second latency, then feeds the same Delta Sync index. No bespoke CDC pipeline to build or babysit.</p>
    <svg width="100%" viewBox="0 0 1060 150" role="img" style="font-family: sans-serif;">
      <title>Lakebase Postgres streams change data into Delta with about fifteen second latency, feeding a vector index and RAG agent</title>
      <defs>
        <marker id="cdfai-arr4" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
          <path d="M1 1L9 5L1 9Z" fill="#0b2026"/>
        </marker>
      </defs>
      <rect x="18" y="40" width="166" height="62" rx="6" fill="rgba(66,153,224,0.10)" stroke="#4299E0" stroke-width="1.8"/>
      <text x="101" y="66" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Lakebase Postgres</text>
      <text x="101" y="86" text-anchor="middle" font-size="14" fill="#618794">orders, customers</text>
      <path d="M184 71 L241 71" fill="none" stroke="#0b2026" stroke-width="1.8" marker-end="url(#cdfai-arr4)"/>
      <text x="213" y="58" text-anchor="middle" font-size="14" font-weight="700" fill="#FF5F46">~15s</text>
      <rect x="259" y="40" width="166" height="62" rx="6" fill="rgba(255,95,70,0.12)" stroke="#FF5F46" stroke-width="1.8"/>
      <text x="342" y="66" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Lakebase CDF</text>
      <text x="342" y="86" text-anchor="middle" font-size="14" fill="#618794">change stream</text>
      <path d="M425 71 L482 71" fill="none" stroke="#0b2026" stroke-width="1.8" marker-end="url(#cdfai-arr4)"/>
      <rect x="500" y="40" width="166" height="62" rx="6" fill="#fff" stroke="#0b2026" stroke-width="1.3"/>
      <text x="583" y="66" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Delta table</text>
      <text x="583" y="86" text-anchor="middle" font-size="14" fill="#618794">in Unity Catalog</text>
      <path d="M666 71 L723 71" fill="none" stroke="#0b2026" stroke-width="1.8" marker-end="url(#cdfai-arr4)"/>
      <rect x="741" y="40" width="130" height="62" rx="6" fill="rgba(66,153,224,0.10)" stroke="#4299E0" stroke-width="1.8"/>
      <text x="806" y="66" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Vector index</text>
      <text x="806" y="86" text-anchor="middle" font-size="14" fill="#618794">Delta Sync</text>
      <path d="M871 71 L928 71" fill="none" stroke="#0b2026" stroke-width="1.8" marker-end="url(#cdfai-arr4)"/>
      <rect x="946" y="40" width="96" height="62" rx="6" fill="rgba(255,95,70,0.12)" stroke="#FF5F46" stroke-width="1.8"/>
      <text x="994" y="66" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Agent</text>
      <text x="994" y="86" text-anchor="middle" font-size="14" fill="#618794">answers</text>
    </svg>

  </div>
</div>

### A6. Search Modes: Semantic, Keyword, and Hybrid

Once an index exists, you can query it three ways — **semantic** (vector), **keyword**, or **hybrid**. The diagram traces one query through all three paths; click a card below to spotlight a single path.
<div class="aism">
  <style>
    .aism { font-family: -apple-system, "Segoe UI", Roboto, sans-serif; color: #0b2026; width: 100%; max-width: 100%; margin: 0; padding: 4px 0; box-sizing: border-box; }
    .aism .diagram { width: 90%; max-width: 100%; margin: 0 auto; box-sizing: border-box; }
    .aism .grid { display: flex; flex-wrap: nowrap; gap: 18px; margin-top: 18px; width: 100%; }
    .aism .card { flex: 1 1 0; min-width: 0; background: #fff; border-radius: 14px; box-shadow: 0 2px 8px rgba(27,49,57,0.07); position: relative; padding: 16px 18px 18px; overflow: hidden; box-sizing: border-box; text-align: left; font: inherit; color: inherit; cursor: pointer; transition: box-shadow .15s ease, opacity .15s ease; }
    .aism .c1 { border: 1.5px solid #C6DCEA; }
    .aism .c2 { border: 1.5px solid #C6DCEA; }
    .aism .c3 { border: 1.5px solid #C6DCEA; }
    .aism .card h3 { margin: 6px 0 4px 0; font-size: 17px; font-weight: 600; }
    .aism .sub { font-size: 14px; color: #1B5162; margin: 0 0 12px 0; }
    .aism ul { margin: 0; padding-left: 18px; }
    .aism li { font-size: 14px; margin-bottom: 11px; line-height: 1.45; }
    .aism code { font-family: ui-monospace, Menlo, Consolas, monospace; font-size: 14px; background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }
    .aism svg { display: block; width: 100%; height: auto; margin: 0 auto; }
    .aism .hint { font-size: 14px; color: #618794; text-align: center; margin: 10px 0 0 0; }
    /* --- focus states: only active once a card is picked --- */
    .aism .card.is-dim { opacity: .55; }
    .aism .card.is-on { box-shadow: 0 0 0 3px rgba(34,114,180,.35), 0 2px 8px rgba(27,49,57,.10); }
    .aism g[data-part] { transition: opacity .18s ease; }
    .aism[data-focus] g[data-part] { opacity: .12; }
    .aism[data-focus="vector"] g[data-part~="vector"] { opacity: 1; }
    .aism[data-focus="keyword"] g[data-part~="keyword"] { opacity: 1; }
    .aism[data-focus="hybrid"] g[data-part~="hybrid"] { opacity: 1; }
    @media (max-width: 900px) { .aism .diagram { width: 100%; } .aism .grid { flex-wrap: wrap; } .aism .card { flex: 1 1 280px; } }
  </style>
  <div class="diagram">
    <svg viewBox="0 208 1000 262" preserveAspectRatio="xMidYMid meet" role="img" aria-label="One shared query splits into vector search using HNSW approximate nearest neighbor and keyword search using Okapi BM25, then both merge into a hybrid step that fuses them with reciprocal rank fusion into one ranked list.">
      <defs>
        <marker id="aismV" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
          <path d="M 0 0 L 10 5 L 0 10 z" fill="#4299E0"></path>
        </marker>
        <marker id="aismK" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
          <path d="M 0 0 L 10 5 L 0 10 z" fill="#FF5F46"></path>
        </marker>
        <marker id="aismG" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
          <path d="M 0 0 L 10 5 L 0 10 z" fill="#7B61FF"></path>
        </marker>
        <marker id="aismA" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
          <path d="M 0 0 L 10 5 L 0 10 z" fill="#FFAB00"></path>
        </marker>
      </defs>
      <!-- ===== QUERY (shared by every path) ===== -->
      <g data-part="vector keyword hybrid">
        <rect x="8" y="302" width="160" height="76" rx="8" fill="#F9F7F4" stroke="#1B5162" stroke-width="2"></rect>
        <text x="88" y="333" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Query</text>
        <text x="88" y="356" text-anchor="middle" font-size="14" fill="#1B5162">text + embedding</text>
        <rect x="8" y="392" width="160" height="48" rx="6" fill="#fff" stroke="#1B5162" stroke-width="1.5" stroke-dasharray="6 4"></rect>
        <text x="88" y="413" text-anchor="middle" font-size="14" font-weight="600" font-family="ui-monospace, Menlo, Consolas, monospace" fill="#0b2026">&#8220;laptop won&#8217;t</text>
        <text x="88" y="431" text-anchor="middle" font-size="14" font-weight="600" font-family="ui-monospace, Menlo, Consolas, monospace" fill="#0b2026">power on&#8221;</text>
      </g>
      <!-- ===== VECTOR BRANCH ===== -->
      <g data-part="vector hybrid">
        <path d="M 168 340 C 216 340, 220 256, 268 256" fill="none" stroke="#4299E0" stroke-width="2" marker-end="url(#aismV)"></path>
        <rect x="272" y="220" width="248" height="72" rx="8" fill="rgba(66,153,224,0.10)" stroke="#4299E0" stroke-width="2" stroke-dasharray="8 5"></rect>
        <text x="292" y="248" font-size="16" font-weight="600" fill="#0b2026">Vector (semantic)</text>
        <text x="292" y="272" font-size="14" fill="#2272B4">HNSW ANN &#183; nearest by distance</text>
      </g>
      <!-- ===== KEYWORD BRANCH ===== -->
      <g data-part="keyword hybrid">
        <path d="M 168 340 C 216 340, 220 424, 268 424" fill="none" stroke="#FF5F46" stroke-width="2" marker-end="url(#aismK)"></path>
        <rect x="272" y="388" width="248" height="72" rx="8" fill="rgba(255,95,70,0.10)" stroke="#FF5F46" stroke-width="2" stroke-dasharray="8 5"></rect>
        <text x="292" y="416" font-size="16" font-weight="600" fill="#0b2026">Keyword (full-text)</text>
        <text x="292" y="440" font-size="14" fill="#FF5F46">Okapi BM25 term relevance</text>
      </g>
      <!-- ===== MERGE + RANKED LIST (hybrid only) ===== -->
      <g data-part="hybrid">
        <path d="M 520 256 C 570 256, 562 340, 612 340" fill="none" stroke="#7B61FF" stroke-width="2" marker-end="url(#aismG)"></path>
        <path d="M 520 424 C 570 424, 562 340, 612 340" fill="none" stroke="#7B61FF" stroke-width="2" marker-end="url(#aismG)"></path>
        <rect x="616" y="302" width="160" height="76" rx="8" fill="rgba(123,97,255,0.10)" stroke="#7B61FF" stroke-width="2" stroke-dasharray="8 5"></rect>
        <text x="696" y="333" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Hybrid</text>
        <text x="696" y="356" text-anchor="middle" font-size="14" fill="#7B61FF">combine with RRF</text>
        <path d="M 776 340 L 824 340" fill="none" stroke="#FFAB00" stroke-width="2" marker-end="url(#aismA)"></path>
        <rect x="828" y="288" width="164" height="104" rx="8" fill="rgba(255,171,0,0.12)" stroke="#FFAB00" stroke-width="2"></rect>
        <text x="910" y="314" text-anchor="middle" font-size="15" font-weight="600" fill="#0b2026">One ranked list</text>
        <rect x="848" y="326" width="124" height="14" rx="4" fill="#FFAB00"></rect>
        <rect x="848" y="346" width="102" height="14" rx="4" fill="#FFCC66"></rect>
        <rect x="848" y="366" width="84" height="14" rx="4" fill="#FFCC66"></rect>
      </g>
    </svg>
  </div>
  <div class="grid">
    <div class="card c1" data-focus-btn="vector" aria-pressed="false">
      <div style="height:5px; border-radius:5px; margin:-4px 0 12px; background:#4299E0;"></div>
      <h3>Vector (semantic) search</h3>
      <p class="sub">Approximate nearest neighbor (HNSW)</p>
      <ul>
        <li>Finds rows whose <strong>meaning</strong> is closest &mdash; even with no shared words.</li>
        <li>The query is embedded; the index returns nearest vectors via <strong>HNSW ANN</strong>.</li>
        <li>Relevance from L2 distance, scored <code>1 / (1 + distance&#178;)</code> &mdash; rank-equivalent to cosine with normalized vectors.</li>
      </ul>
    </div>
    <div class="card c2" data-focus-btn="keyword" aria-pressed="false">
      <div style="height:5px; border-radius:5px; margin:-4px 0 12px; background:#FF5F46;"></div>
      <h3>Keyword (full-text) search</h3>
      <p class="sub">Okapi BM25 relevance</p>
      <ul>
        <li>Ranks rows by how well they match your <strong>exact terms</strong>.</li>
        <li>Uses <strong>Okapi BM25</strong> scoring across text columns.</li>
        <li>Best for names, codes, SKUs, and exact-term lookups where wording matters.</li>
      </ul>
    </div>
    <div class="card c3" data-focus-btn="hybrid" aria-pressed="false">
      <div style="height:5px; border-radius:5px; margin:-4px 0 12px; background:#7B61FF;"></div>
      <h3>Hybrid search</h3>
      <p class="sub">Runs both, merges the results</p>
      <ul>
        <li>Combines both result sets into <strong>one ranked list</strong>.</li>
        <li>Merges with <strong>Reciprocal Rank Fusion (RRF)</strong>.</li>
        <li>Best when queries mix intent with specific terms &mdash; a strong default.</li>
      </ul>
    </div>
  </div>
  <p class="hint">Click a card to spotlight that path in the diagram &mdash; click it again to show everything.</p>
  <script>
  (function () {
    var root = document.currentScript.closest('.aism');
    if (!root) { return; }
    var btns = root.querySelectorAll('[data-focus-btn]');
    function apply(mode) {
      if (mode) { root.setAttribute('data-focus', mode); } else { root.removeAttribute('data-focus'); }
      btns.forEach(function (b) {
        var on = mode && b.getAttribute('data-focus-btn') === mode;
        b.setAttribute('aria-pressed', on ? 'true' : 'false');
        b.classList.toggle('is-on', !!on);
        b.classList.toggle('is-dim', !!mode && !on);
      });
    }
    btns.forEach(function (b) {
      b.addEventListener('click', function () {
        var mode = b.getAttribute('data-focus-btn');
        apply(root.getAttribute('data-focus') === mode ? null : mode);
      });
    });
  })();
  </script>
</div>

## B. Managed vs. Custom RAG

### B1. Two Paths to RAG

Now that you've built the full custom pipeline (parse, chunk, index, search), let's zoom out. Databricks provides two complementary approaches to building retrieval systems:

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 780 320" role="img" style="font-family: sans-serif;">
  <title>Managed RAG vs Custom RAG: two paths from documents to answers</title>
  <defs>
    <marker id="rag-arrow-green" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
    <marker id="rag-arrow-blue" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Documents (shared start) -->
  <rect x="10" y="110" width="130" height="90" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="75" y="140" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Documents</text>
  <text x="75" y="160" text-anchor="middle" font-size="13" fill="#618794">PDFs, tables,</text>
  <text x="75" y="176" text-anchor="middle" font-size="13" fill="#618794">UC Volumes</text>

  <!-- Fork arrows -->
  <path d="M140 135 L210 65" fill="none" stroke="#FF5F46" stroke-width="1.8" marker-end="url(#rag-arrow-green)"/>
  <path d="M140 175 L210 235" fill="none" stroke="#2272B4" stroke-width="1.8" marker-end="url(#rag-arrow-blue)"/>

  <!-- MANAGED PATH (top) -->
  <rect x="215" y="15" width="310" height="100" rx="10" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="370" y="42" text-anchor="middle" font-size="17" font-weight="700" fill="#FF5F46">Managed RAG</text>
  <text x="370" y="62" text-anchor="middle" font-size="14" fill="#0b2026">Genie Agent</text>
  <text x="370" y="82" text-anchor="middle" font-size="13" fill="#618794">Parsing, chunking, embedding,</text>
  <text x="370" y="98" text-anchor="middle" font-size="13" fill="#618794">indexing, serving are automatically implemented</text>

  <!-- CUSTOM PATH (bottom) -->
  <rect x="215" y="190" width="310" height="110" rx="10" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="370" y="218" text-anchor="middle" font-size="17" font-weight="700" fill="#2272B4">Custom RAG</text>
  <text x="370" y="238" text-anchor="middle" font-size="14" fill="#0b2026">AI SQL Functions + Additional Logic</text>
  <text x="370" y="258" text-anchor="middle" font-size="13" fill="#618794">ai_parse_document → ai_prep_search</text>
  <text x="370" y="274" text-anchor="middle" font-size="13" fill="#618794">→ Delta table → AI Search index</text>
  <text x="370" y="290" text-anchor="middle" font-size="13" fill="#618794">Full control over every component</text>

  <!-- Converge arrows -->
  <path d="M525 65 L600 135" fill="none" stroke="#FF5F46" stroke-width="1.8" marker-end="url(#rag-arrow-green)"/>
  <path d="M525 245 L600 175" fill="none" stroke="#2272B4" stroke-width="1.8" marker-end="url(#rag-arrow-blue)"/>

  <!-- Answers (shared end) -->
  <rect x="605" y="110" width="160" height="90" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="685" y="140" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Answers</text>
  <text x="685" y="160" text-anchor="middle" font-size="13" fill="#618794">LLM + retrieved</text>
  <text x="685" y="176" text-anchor="middle" font-size="13" fill="#618794">context</text>
</svg>
</div>

**Managed RAG (Genie Agents):**
- You provide documents (in a UC Volume or AI Search Index)
- The system handles parsing, chunking, embedding, indexing, and serving
- Optimized automatically via feedback loops
- Best for: standard RAG use cases where speed to production matters

**Custom RAG (Agent Framework + AI Search):**
- You build the pipeline yourself with full ownership of each step
- Full control over parsing strategy, chunking approach, embedding model, and retrieval logic
- Use AI Search indexes as retrieval tools in your custom agent
- Best for: unique requirements, complex multi-step reasoning, novel architectures

### B2. Governance with Unity Catalog

AI Search indexes are governed by Unity Catalog. Access to Unity Catalog objects in the retrieval pipeline is controlled with Unity Catalog privileges, while AI search endpoints are controlled with endpoint ACLs. Audit visibility is available through system tables, and lineage can be captured across Unity Catalog tables and paths used in the retrieval pipeline.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 820 200" role="img" style="font-family: sans-serif;">
 <title>Governance across the retrieval pipeline with Unity Catalog and AI search endpoint ACLs</title>
 <defs>
 <marker id="uc-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
 <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
 </marker>
 </defs>

 <!-- Governance banner -->
 <rect x="10" y="10" width="800" height="180" rx="10" fill="none" stroke="#1B3139" stroke-width="1" stroke-opacity="0.3"/>
 <text x="410" y="32" text-anchor="middle" font-size="14" font-weight="700" fill="#1B3139">Governance across the retrieval pipeline</text>

 <!-- UC Volume -->
 <rect x="30" y="58" width="120" height="56" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.2"/>
 <text x="90" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">UC Volume</text>
 <text x="90" y="100" text-anchor="middle" font-size="12" fill="#618794">Source PDFs</text>

 <path d="M150 86 L185 86" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#uc-arrow)"/>

 <!-- Delta Table -->
 <rect x="190" y="58" width="145" height="56" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.2"/>
 <text x="262.5" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Delta Table</text>
 <text x="262.5" y="100" text-anchor="middle" font-size="12" fill="#618794">Chunks + metadata</text>

 <path d="M335 86 L370 86" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#uc-arrow)"/>

 <!-- VS Index -->
 <rect x="375" y="58" width="135" height="56" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.2"/>
 <text x="442.5" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">VS Index</text>
 <text x="442.5" y="100" text-anchor="middle" font-size="12" fill="#618794">Vectors + synced cols</text>

 <path d="M510 86 L555 86" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#uc-arrow)"/>

 <!-- VS Endpoint -->
 <rect x="560" y="58" width="120" height="56" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.2"/>
 <text x="620" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">VS Endpoint</text>
 <text x="620" y="100" text-anchor="middle" font-size="12" fill="#618794">Query interface</text>

 <path d="M680 86 L715 86" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#uc-arrow)"/>

 <!-- KA / Agent -->
 <rect x="720" y="58" width="70" height="56" rx="8" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.2"/>
 <text x="755" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Agent</text>
 <text x="755" y="100" text-anchor="middle" font-size="12" fill="#618794">RAG</text>

 <!-- Governance labels -->
 <g>
 <text x="90" y="132" text-anchor="middle" font-size="12" fill="#618794">UC privileges</text>
 <text x="262.5" y="132" text-anchor="middle" font-size="12" fill="#618794">UC privileges</text>
 <text x="442.5" y="132" text-anchor="middle" font-size="12" fill="#618794">UC privileges</text>
 <text x="620" y="132" text-anchor="middle" font-size="12" fill="#618794">Endpoint ACLs</text>
 <text x="755" y="132" text-anchor="middle" font-size="12" fill="#618794">Queries via</text>
 <text x="755" y="146" text-anchor="middle" font-size="12" fill="#618794">endpoint</text>

</svg>
</div>

## C. AI Search vs. Lakebase Search

Databricks now offers two search options for retrieval. They complement each other and serve different architectures:

|  | **AI Search** | **Lakebase Search** |
|---|---|---|
| **What it is** | Managed search service over Unity Catalog tables and volumes — handles embedding, indexing, and retrieval as a fully managed stack | Postgres extensions adding vector + keyword search inside a Lakebase project |
| **Primary use case** | Production search — search bars, recommendation systems, and document RAG for unstructured content (PDFs, docs, slides) | Operational search — query structured and semi-structured data already in Postgres |
| **Used by** | Search bars, recommender systems, Genie Agents, custom RAG agents, any application calling the AI Search API | Applications backed by Lakebase — customer-facing apps, internal tools, agent memory |
| **Search modes** | Vector (ANN), keyword (BM25), hybrid (RRF), and hybrid + reranker | Vector (ANN), keyword (BM25), and hybrid (RRF fusion) |
| **JOINs with operational data** | No — results come back as a separate retrieval call | Yes — search, JOIN, and filter in one SQL query, one transaction |
| **Permissions model** | Unity Catalog ACLs on the index | Postgres row-level GRANTs on the same tables |
| **Index management** | Fully managed by Databricks | You create indexes with `CREATE INDEX ... USING lakebase_ann / lakebase_bm25` |
| **Lakehouse integration** | Native — indexes sync from UC tables/volumes via Delta Sync | Via synced tables — sync a UC table into Lakebase with type mapping for search columns |

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">When to choose which</strong>
<p style="margin: 8px 0 0 0; color: #333;">
Use <strong>AI Search</strong> when you want a fully managed search engine — search bars, recommendations, or document RAG — with built-in hybrid retrieval, reranking, and quality evaluation out of the box. Use <strong>Lakebase Search</strong> when your data already lives in Lakebase and you want low-latency search co-located with your operational tables — for example, searching customer reviews, product catalogs, or support tickets alongside structured metadata in a single SQL query.
</p>
</div>

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">

  <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>

  <img src="https://files.training.databricks.com/binder/prod_main/building-rag-agents-with-agent-bricks-en_us-2.0.1/images/20261116T191026Z/Building RAG Agents with Agent Bricks/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to learn more about AI Search and embedding strategies? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/building-rag-agents-with-agent-bricks-en_us-2.0.1/images/20261116T191026Z/Building RAG Agents with Agent Bricks/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #F9F7F4; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">How does Databricks AI Search handle hybrid keyword and semantic search?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
